In [ ]:
# Import libraries
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import json

from utils.seed import set_random_seed
from utils.constants import MIMIC_RARE_ICD_CODES

In [ ]:
primary_dir = "data/MIMICIII_last48h_ts2h"
time_resolution = "2h"	

ts_dir = f"data/MIMICIII_last48h_ts{time_resolution}/timeseries"
split_dir = f'data/MIMICIII_last48h_ts{time_resolution}/split'
primary_split_dir = os.path.join(primary_dir, "split")

Path(split_dir, 'all').mkdir(parents=True, exist_ok=True)

set_random_seed(1)

## Define the rare diseases

1. Calculate the count statistics of the Diseases
2. Exclude diseases with subject_id number < 10.
3. Separate the diseases into 10 groups sorted by disease subject counts.

In [ ]:
# Load data
demo_df = pd.read_csv(os.path.join(ts_dir, "demographics.csv"), sep=',')
ts_df = pd.read_csv(os.path.join(ts_dir, "time-series.csv"), sep=',')
label_df = pd.read_csv(os.path.join(ts_dir, "label.csv"), sep=',')

# Calculate sample size threshold
threshold = len(demo_df) / 2500

# Identify ICD-9 codes that meet the condition
icd9_counts = demo_df.groupby('icd9_code')['subject_id'].nunique().sort_values(ascending=False)

# Create a dictionary to map each target ICD-9 code to its unique subject IDs
icd92subject_id = {}
for icd9_code in demo_df.icd9_code.unique():
    subject_ids = demo_df[demo_df.icd9_code == icd9_code].subject_id.unique()
    icd92subject_id[icd9_code] = subject_ids

# Print the number of unique subject IDs for each ICD-9 code
icd9_count_df = pd.DataFrame([[key, len(value)] for key, value in icd92subject_id.items()], columns=['icd9_code', 'count'])
icd9_count_df = icd9_count_df.sort_values('count', ascending=False)

# save the target ICD-9 count df to a csv file
icd9_count_df.to_csv(os.path.join(ts_dir, "icd9_count.csv"), index=False)
icd9_count_df.head()


In [ ]:
# Find the index of the first ICD-9 code with count less than or equal to the threshold
threshold_position = next((i for i, count in enumerate(icd9_counts.values) if count <= threshold), len(icd9_counts))

# Plot the Count distribution with a log scale on the y-axis
plt.figure(figsize=(12, 6))
plt.bar(range(len(icd9_counts)), icd9_counts.values)
plt.yscale('log')  # Set y-axis to logarithmic scale

# Add a vertical line at the threshold position
plt.axvline(x=threshold_position, color='red', linestyle='--', linewidth=1.5, label=f'Count Threshold (1/2500) Position = {threshold_position}')

plt.xlabel('i^th ICD-9 Code Ordered by Count')
plt.ylabel('Count')
plt.title('Count Distribution of ICD-9 Codes (Log Scale)')
plt.legend()

# Set x-ticks at intervals of 100 (e.g., 100th, 200th) and label them
interval = 100
ticks = list(range(0, len(icd9_counts), interval))
plt.xticks(ticks=ticks, labels=ticks)

plt.tight_layout()
plt.show()


### Get the dataset after exclude diseases with less than 10 patients

In [ ]:
print(f"Number of unique ICD-9 codes: {len(demo_df.icd9_code.unique())}")

# Exclude ICD-9 codes with count less than 10
icd9_counts = demo_df.groupby('icd9_code')['subject_id'].nunique().sort_values(ascending=False)
icd9_counts = icd9_counts[icd9_counts >= 10]
include_icd9 = icd9_counts.index
demo_df = demo_df[demo_df.icd9_code.isin(include_icd9)]
subject_ids = demo_df['subject_id'].unique()
ts_df = ts_df[ts_df['subject_id'].isin(subject_ids)]
label_df = label_df[label_df['subject_id'].isin(subject_ids)]
print(f"Number of unique ICD-9 codes after excluding diseases with count less than 10: {len(demo_df.icd9_code.unique())}")

# Exclude diseases that have at least one label with all negative entries
# excluded_diseases = []
# label_columns = ['days_90_expire_flag']
# unique_disease_codes = demo_df['icd9_code'].unique()

# for disease_code in unique_disease_codes:
#     # Get the subset of subject IDs for the current disease
#     disease_hadm_ids = demo_df.loc[demo_df['icd9_code'] == disease_code, 'hadm_id'].unique()
#     # Filter label_df for these subject IDs
#     disease_label_df = label_df[label_df['hadm_id'].isin(disease_hadm_ids)]
    
#     # Check if any label column has only zeros for the disease
#     if any(disease_label_df[label].sum() == 0 for label in label_columns):
#         excluded_diseases.append(disease_code)

# Filter out diseases with any label column having all negative labels
# demo_df = demo_df[~demo_df['icd9_code'].isin(excluded_diseases)]
# subject_ids = demo_df['subject_id'].unique()
# ts_df = ts_df[ts_df['subject_id'].isin(subject_ids)]
# label_df = label_df[label_df['subject_id'].isin(subject_ids)]
# print(f"Number of unique ICD-9 codes after excluding diseases with any all-negative labels: {demo_df['icd9_code'].nunique()}")

# calculate the sample size threshold
threshold = len(demo_df) / 2500
icd9_counts = demo_df.groupby('icd9_code')['subject_id'].nunique().sort_values(ascending=False)

# Create a dictionary to map each target ICD-9 code to its unique subject IDs
icd92subject_id = {}
for icd9_code in demo_df.icd9_code.unique():
    subject_ids = demo_df[demo_df.icd9_code == icd9_code].subject_id.unique()
    icd92subject_id[icd9_code] = subject_ids

# save the demo_df to a csv file
demo_df.to_csv(os.path.join(split_dir, 'all', "demographics.csv"), index=False)
ts_df.to_csv(os.path.join(split_dir, 'all', "time-series.csv"), index=False)
label_df.to_csv(os.path.join(split_dir, 'all', "label.csv"), index=False)


In [ ]:
# Find the index of the first ICD-9 code with count less than or equal to the threshold
threshold_position = next((i for i, count in enumerate(icd9_counts.values) if count <= threshold), len(icd9_counts))

# Plot the Count distribution with a log scale on the y-axis
plt.figure(figsize=(12, 6))
plt.bar(range(len(icd9_counts)), icd9_counts.values)
plt.yscale('log')  # Set y-axis to logarithmic scale

# Add a vertical line at the threshold position
plt.axvline(x=threshold_position, color='red', linestyle='--', linewidth=1.5, label=f'Count Threshold (1/2500) Position = {threshold_position}')

plt.xlabel('i^th ICD-9 Code Ordered by Count')
plt.ylabel('Count')
plt.title('Count Distribution of ICD-9 Codes (Log Scale)')
plt.legend()

# Set x-ticks at intervals of 100 (e.g., 100th, 200th) and label them
interval = 100
ticks = list(range(0, len(icd9_counts), interval))
plt.xticks(ticks=ticks, labels=ticks)

plt.tight_layout()
plt.show()

## Split the dataset

1. Split into train:val:test with the ratio of 4:1:1 based on subject_id.

In [ ]:
def get_subset_ids(df_path, id_col, target_diseases, disease_col='icd9_code', labels=None):
    """
    This function splits a dataset of patient records into training, validation, and test sets according to a specified ratio (4:1:1).
    The function ensures that the test set contains at least one positive sample for each specified label combination to facilitate
    robust evaluation. 

    Parameters:
    - df_path (str): Path to the CSV file containing the dataset.
    - id_col (str): Column name for the unique patient or record identifier (e.g., 'hadm_id').
    - disease_col (str): Column name for the disease or main category (e.g., 'icd9_code').
    - labels (dict): A dictionary specifying the labels that must have at least one positive sample in the test set.
      Format: `{disease_id: {hadm_id: (label1, label2, ...), ...}, ...}`
        - `disease_id`: The main category (e.g., disease type) matching `disease_col` in the dataframe.
        - `hadm_id`: The unique identifier for each patient or record, matching `id_col` in the dataframe.
        - `(label1, label2, ...)`: A tuple of binary labels (0 or 1), where each element represents a specific condition or 
          attribute that must be present in at least one sample in the test set. At least one label in the tuple should be 1
          (positive) for the sample to be considered a positive sample.

    Returns:
    - dict: A dictionary with keys "train", "val", and "test", where each key contains a list of patient IDs belonging to 
            each respective subset.
        - "train": List of patient IDs assigned to the training set.
        - "val": List of patient IDs assigned to the validation set.
        - "test": List of patient IDs assigned to the test set, with at least one positive sample per specified label.
    """

    # Load the dataset
    df = pd.read_csv(df_path)
    
    # Step 1: Define lists to hold patient IDs for each set
    train_ids, val_ids, test_ids = [], [], []
    test_ids_count = {}  # Dictionary to keep track of added samples per disease for test set

    # Step 2: Ensure each disease_code has at least one positive sample for each class in the test set
    if labels is not None:
        for disease_code, subject_dict in labels.items():
            if disease_code not in target_diseases:
                continue
            else:
                class0_added, class1_added = False, False  # Flags to ensure one positive sample per class is added to the test set
                for subject_id, label_tuple in subject_dict.items():
                    # Filter rows that match both the disease_code and hadm_id in the dataset
                    label_group = df[(df[disease_col] == disease_code) & (df[id_col] == subject_id)]
                    label_ids = label_group[id_col].unique() # may contain multiple hadm_ids
                    
                    # Check if label_tuple has a positive label for class 1
                    if label_tuple[0] == 1 and label_tuple[1] == 1:
                        # If there are matching samples, add one to the test set
                        if len(label_ids) > 0:
                            test_ids.extend(label_ids)
                            class0_added = True
                            class1_added = True
                    
                            # Remove the selected test sample from the original dataset to avoid data leakage
                            df = df[~(df[id_col] == label_ids[0])]
                    elif label_tuple[0] == 1 and not class0_added:
                        # If there are matching samples, add one to the test set
                        if len(label_ids) > 0:
                            test_ids.extend(label_ids)
                            class0_added = True
                            
                            # Remove the selected test sample from the original dataset to avoid data leakage
                            df = df[~(df[id_col] == label_ids[0])]
                    elif label_tuple[1] == 1 and not class1_added:
                        # If there are matching samples, add one to the test set
                        if len(label_ids) > 0:
                            test_ids.extend(label_ids)
                            class1_added = True
                            
                            # Remove the selected test sample from the original dataset to avoid data leakage
                            df = df[~(df[id_col] == label_ids[0])]

                    if class1_added and class0_added:
                        break

            # Track the count of added test samples for this disease_code
            test_ids_count[disease_code] = int(class0_added) + int(class1_added)
            assert class0_added and class1_added , f"Could not find positive samples for both classes in test set for disease {disease_code}"
            
    # Step 3: Group remaining data by disease and split each group individually
    assigned_ids = set(test_ids)  # Start with test_ids if they were pre-assigned
    disease_groups = df.groupby(disease_col)

    for disease, group in disease_groups:
        # if labels is not None and disease in labels:
        #     # Skip diseases that already have at least one positive sample for each class in the test set
        #     test_num_offset = len(test_ids)
        # else:
        #     test_num_offset = 0
        # Get unique patient IDs for the current disease
        unique_ids = group[id_col].unique()
        np.random.shuffle(unique_ids)
        
        # Remove any IDs already assigned to prevent duplicates
        unique_ids = [int(uid) for uid in unique_ids if uid not in assigned_ids]
        
        # Define train, val, and test split indices
        test_count = test_ids_count.get(disease, 0)
        train_num = int(len(unique_ids) * 4 / 6) + test_count//2
        val_num = int(len(unique_ids) / 6) # avoid negative values
        
        # Assign IDs to train, val, and test sets
        train_ids.extend(unique_ids[:train_num])
        val_ids.extend(unique_ids[train_num:train_num + val_num])
        test_ids.extend(unique_ids[train_num+val_num:])
        test_ids = [int(uid) for uid in test_ids]
        
        # Update assigned_ids to include newly added IDs
        assigned_ids.update(unique_ids)
    
    assert len(set(train_ids).intersection(set(val_ids))) == 0, "Train and val sets have overlapping IDs"
    assert len(set(train_ids).intersection(set(test_ids))) == 0, "Train and test sets have overlapping IDs"
    assert len(set(val_ids).intersection(set(test_ids))) == 0, "Val and test sets have overlapping IDs"
    # Return the combined sets
    return {
        "train": train_ids, 
        "val": val_ids,
        "test": test_ids
    }

# dump the train, val, test ids to a csv file
def dump_subsets(df_path, 
                 id_col,
                 set_id_dict, 
                 split_dir):
    df = pd.read_csv(df_path)
    df_name = os.path.basename(df_path)
    for set_name, idx in set_id_dict.items():
        set_dir = os.path.join(split_dir, set_name)
        Path(set_dir).mkdir(parents=True, exist_ok=True)
        
        subset_df = df[df[id_col].isin(idx)]
        subset_df.to_csv(os.path.join(set_dir, df_name), index=False)
    
    # save json
    with open(os.path.join(split_dir, f"split_id.json"), 'w') as f:
        json.dump(set_id_dict, f, indent=4)
    

### 90 days Mortality and 30 days Readmission

In [ ]:
# mortality
demo_path = os.path.join(split_dir, 'all', "demographics.csv")
label_path = os.path.join(split_dir, 'all', "label.csv")
ts_path = os.path.join(split_dir, 'all', "time-series.csv")
data_files = [demo_path, label_path, ts_path]
id_col = 'subject_id'

# load labels
demo_df = pd.read_csv(demo_path)
label_df = pd.read_csv(label_path)
labels = {}
for disease_code in MIMIC_RARE_ICD_CODES:
    # Get unique subject IDs associated with the current disease code
    hadm_ids = demo_df[demo_df['icd9_code'] == disease_code]['hadm_id'].unique()
    # Filter the label dataframe for these subject IDs
    _label_df = label_df[label_df['hadm_id'].isin(hadm_ids)]
    # Aggregate labels for each subject_id
    subject_labels = {}
    for subject_id, group in _label_df.groupby('subject_id'):
        # Use max to check if there is any positive label for each flag across multiple rows for the same subject
        days_90_expire = group['days_90_expire_flag'].max()
        days_30_readmission = group['days_30_readmission_flag'].max()
        subject_labels[subject_id] = (days_90_expire, days_30_readmission)
    
    # Store the aggregated labels for each disease code
    labels[disease_code] = subject_labels

# # Split the dataset into train, validation, and test sets
if split_dir != primary_split_dir:
    print(f"Use split_id from the {primary_split_dir}")
    with open(os.path.join(primary_split_dir, f"split_id.json"), 'r') as f:
        set_id_dict = json.load(f)
else:
    print(f"Generate new split_id.json from {demo_path}")
    set_id_dict = get_subset_ids(demo_path, id_col, target_diseases=MIMIC_RARE_ICD_CODES, labels=labels)

for df_path in data_files:
    dump_subsets(df_path, id_col, set_id_dict, split_dir)

In [ ]:
# check no overlap between train, val, test sets
train_ids = set(set_id_dict['train'])
val_ids = set(set_id_dict['val'])
test_ids = set(set_id_dict['test'])

assert len(train_ids.intersection(val_ids)) == 0, "Overlap between train and val sets"
assert len(train_ids.intersection(test_ids)) == 0, "Overlap between train and test sets"
assert len(val_ids.intersection(test_ids)) == 0, "Overlap between val and test sets"

# check the positive samples in the test set
test_label_df = pd.read_csv(os.path.join(split_dir, "test/label.csv"))
for disease_code in MIMIC_RARE_ICD_CODES:
    hadm_ids = demo_df[demo_df['icd9_code'] == disease_code]['hadm_id'].values
    _label_df = test_label_df[test_label_df['hadm_id'].isin(hadm_ids)]
    print(f"{len(_label_df)} Samples for {disease_code} in the test set, with {_label_df['days_90_expire_flag'].sum()} mortality and {_label_df['days_30_readmission_flag'].sum()} Readmission.")
    assert _label_df['days_90_expire_flag'].sum() > 0, f"No positive mortality samples for {disease_code} in the test set"
    assert _label_df['days_30_readmission_flag'].sum() > 0, f"No positive readmission samples for {disease_code} in the test set"
    